In [ ]:
import pandas as pd
import numpy as np
import yfinance as yfin
import ta

In [ ]:
# Descargamos datos de USD-JPY 
df = yfin.download('JPY=X',start='2016-01-01', end='2021-01-01') 
df['EMA_5'] = ta.trend.ema_indicator(close=df["Close"], window=5, fillna=True)/df["Close"] 
df['EMA_20'] = ta.trend.ema_indicator(close=df["Close"], window=20, fillna=True)/df["Close"] 
df['EMA_50'] = ta.trend.ema_indicator(close=df["Close"], window=50, fillna=True)/df["Close"] 
df['EMA_100'] = ta.trend.ema_indicator(close=df["Close"], window=100, fillna=True)/df["Close"] 
df 

In [ ]:
# RSI
df['RSI'] = ta.momentum.rsi(close=df["Close"], fillna=True) 
# ATR
df['ATR'] = ta.volatility.average_true_range(high=df["High"], low=df["Low"], close=df["Close"], fillna=True) 
# WR
df['WR'] = ta.momentum.williams_r(high=df["High"], low=df["Low"], close=df["Close"], fillna=True) 
df 

In [ ]:
# Crea el Target (1 si el precio sube el día siguiente, -1 si baja) 
target = np.where(np.array(df['Close'][1:]) > np.array(df['Close'][:-1]), 1, -1) 
# Elimina el último día para el que no tenemos info del precio el día siguiente. 
df.drop(df.tail(1).index, inplace=True) 
# Creamos la columna target 
df['Target'] = target 
# Elimina los primeros 29 días dónde los indicadores técnicos no tienen suficiente info 
df.drop(df.head(29).index, inplace=True) 
df.Target[df.Target==1].count() 
df.Target[df.Target==’1].count() 

In [ ]:
# Divida el conjunto de datos en una característica o un conjunto de datos independiente (X) y un destino o conjunto de datos dependiente (Y) 
X = df.drop('Target',axis=1) 
Y = df.Target 
# Vuelva a dividir los datos, pero esta vez en 90% de entrenamiento y 10% de prueba. 
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.3, shuffle=False) 
print(X_train.shape) 

In [ ]:
#Standardization 
ss = StandardScaler() 
X_train = ss.fit_transform(X_train) 
X_test = ss.fit_transform(X_test) 
X_train.mean()

In [ ]:
# Sorte vectorial de clasificación (SVC) con kernerl rbf
svc = SVC(kernel='rbf',random_state=1) 
# Entrenar el modelo de SVC
svc = svc.fit(X_train, Y_train) 
# Curva ROC
plot_roc_curve(svc, X_test, Y_test) 
plt.show()

In [ ]:
# Crear el modelo clasificador de Random Forest 
rfc = RandomForestClassifier(random_state=1) 
# Entrenar el modelo de Random Forest 
rfc = rfc.fit(X_train, Y_train) 
# Curva ROC 
plot_roc_curve(rfc, X_test, Y_test) 
plt.show() 

In [ ]:
# Crear el modelo clasificador de Redes Neuronales 
mlp = MLPClassifier(solver='adam', hidden_layer_sizes=(32,16), max_iter=1000,random_state=1) 
# Entrenar el modelo de Redes Neuronales 
mlp = mlp.fit(X_train, Y_train) 
# Curva ROC
plot_roc_curve(mlp, X_test, Y_test) 
plt.show() 